# Item details — Main Table Exploration

In [0]:
from pyspark.sql import functions as F

# =====================================================
# Load Item Details main table from GCS Raw
# =====================================================

item_details_path = (
    "gs://market-intelligence-raw/ebay/item_details/"
)

item_details_df = (
    spark.read
    .option("multiLine", "false")
    .json(item_details_path + "*.jsonl.gz")
)

# =====================================================
# Basic structural profile
# =====================================================

print("Total rows:", item_details_df.count())
print("Total columns:", len(item_details_df.columns))

item_details_df.printSchema()


# Item Details — Initial Schema Findings

## 1. Dataset Overview

The Item Details dataset was loaded from the raw GCS layer into a PySpark DataFrame.

| Metric | Result |
|---|---:|
| Total Rows | 48,428 (At the time of profiling )
| Total Columns | 111 |
| Source | GCS — `ebay/item_details/` |
| Format | GZIP-compressed JSON Lines |

The dataset contains detailed information about individual eBay listings, including product attributes, pricing, seller information, listing conditions, return policies, and product review ratings.

Compared with the Browse Search dataset, Item Details provides a richer set of attributes for each listing.

## 2. Schema and Column Groups

The 111 columns can be organized into the following logical groups:

| Category | Representative Columns |
|---|---|
| Listing Identity | `item_id`, `legacy_item_id`, `epid` |
| Product Information | `title`, `subtitle`, `description`, `short_description`, `brand`, `mpn`, `gtin` |
| Product Attributes | `color`, `material`, `pattern`, `size`, `size_system`, `age_group`, `gender` |
| Category Information | `category_id`, `category_id_path`, `category_path` |
| Pricing | `price__*`, `current_bid_price__*`, `minimum_price_to_bid__*` |
| Marketing and Discounts | `marketing_price__*` |
| Unit Pricing | `unit_price__*`, `unit_pricing_measure` |
| Seller Information | `seller__username`, `seller__feedback_percentage`, `seller__feedback_score` |
| Listing Dates | `item_creation_date`, `item_end_date` |
| Item Location | `item_location__*` |
| Product Reviews | `primary_product_review_rating__average_rating`, `primary_product_review_rating__review_count` |
| Item Groups | `primary_item_group__*` |
| Images | `image__*`, `primary_item_group__item_group_image__*` |
| Bidding Information | `bid_count`, `unique_bidder_count`, `reserve_price_met` |
| Return Policies | `return_terms__*` |
| Manufacturer Information | `manufacturer__*` |
| Checkout and Listing Flags | `adult_only`, `priority_listing`, `immediate_pay`, `eligible_for_inline_checkout` |
| DLTHub Metadata | `_dlt_id`, `_dlt_load_id` |

The schema includes both core listing attributes and additional information relevant to product intelligence, such as discount details, customer review ratings, bidding activity, and return policies.

## 3. Data Types Observed

The initial schema contains three main Spark data types:

| Data Type | Examples |
|---|---|
| String | Product attributes, identifiers, URLs, dates, pricing values, seller attributes |
| Long | `bid_count`, `unique_bidder_count`, `seller__feedback_score`, image dimensions |
| Boolean | `adult_only`, `reserve_price_met`, `return_terms__returns_accepted`, `priority_listing` |

All 111 columns are marked as nullable in the inferred Spark schema.

This indicates that Spark permits null values in these fields. Actual null counts and field completeness have not yet been measured.

## 4. Pricing Fields Require Type Validation

Several fields representing numerical values are inferred as strings.

Examples include:

- `price__value`
- `current_bid_price__value`
- `minimum_price_to_bid__value`
- `marketing_price__discount_amount__value`
- `marketing_price__original_price__value`
- `marketing_price__discount_percentage`
- `unit_price__value`

These fields will require numeric conversion before performing price comparisons, discount calculations, aggregations, or opportunity scoring.

The corresponding currency fields must also be considered when comparing prices across listings.

## 5. Listing Dates and Product Attributes

The listing creation and end dates are stored as strings:

- `item_creation_date`
- `item_end_date`

These fields will require timestamp parsing before calculating listing age, analyzing listing duration, or performing time-based filtering.

The dataset also contains product attributes such as brand, color, size, material, and manufacturer details. Their availability and completeness should be examined before deciding which attributes can be reliably used for downstream analytics.

## 6. Additional Information Available in Item Details

Compared with Browse Search, this table exposes several additional groups of information:

- **Product reviews:** Average product rating and review count are available through `primary_product_review_rating__*`.

- **Bidding activity:** Bid count, unique bidder count, and reserve-price status provide additional signals for auction listings.

- **Return policies:** Return acceptance, return period, refund method, and shipping-cost responsibility are available under `return_terms__*`.

- **Product grouping:** Primary item-group identifiers, titles, and images provide information about grouped or related listings.

These fields may support richer product-level analysis and opportunity scoring, subject to further completeness and validity checks.


In [0]:
# =====================================================
# Establish record grain and uniqueness
# =====================================================

item_details_df.agg(
    F.count("*").alias("total_rows"),

    F.countDistinct("item_id")
     .alias("distinct_item_ids"),

    F.sum(
        F.when(F.col("item_id").isNull(), 1)
         .otherwise(0)
    ).alias("null_item_ids"),

    F.countDistinct("_dlt_id")
     .alias("distinct_dlt_ids"),

    F.sum(
        F.when(F.col("_dlt_id").isNull(), 1)
         .otherwise(0)
    ).alias("null_dlt_ids")
).show(truncate=False)


## 7. Record Grain and Listing Uniqueness

The Item Details dataset was checked for listing-level uniqueness and ingestion-record completeness using `item_id` and `_dlt_id`.

### 7.1 Uniqueness Check Results

| Metric | Result |
|---|---:|
| Total Rows | 48,428 |
| Distinct Item IDs | 48,428 |
| Null Item IDs | 0 |
| Distinct `_dlt_id` | 48,428 |
| Null `_dlt_id` | 0 |

### 7.2 Findings

- **Unique listing identifiers:** All 48,428 records have distinct `item_id` values. No repeated listing IDs were found in the current Item Details snapshot.

- **Complete listing identifiers:** There are no null values in `item_id`. Every record can be associated with a marketplace listing.

- **Unique ingestion identifiers:** All 48,428 records have distinct `_dlt_id` values, with no null ingestion IDs.

- **Record grain:** The current snapshot follows a one-row-per-listing structure based on `item_id`. Unlike the Browse Search snapshot, no repeated listing observations were identified in this table.

### 7.3 Comparison with Browse Search

| Metric | Browse Search | Item Details |
|---|---:|---:|
| Total Rows | 115,560 | 48,428 |
| Distinct Item IDs | 105,185 | 48,428 |
| Null Item IDs | 0 | 0 |
| Distinct `_dlt_id` | 115,560 | 48,428 |
| Repeated Item IDs | Yes | No |

Browse Search contains multiple observations of certain listings, whereas the current Item Details snapshot contains one record per distinct listing.

This difference is consistent with the different roles of the two datasets: Browse Search captures listing discovery results, while Item Details provides detailed listing-level information.

### 7.4 Implications for Downstream Processing

- `item_id` can serve as the listing-level business key for the current Item Details snapshot.

- `_dlt_id` remains useful for tracking individual ingested records and maintaining ingestion traceability.

- No listing-level deduplication is indicated by the current uniqueness results. However, uniqueness should be validated again after combining data from multiple ingestion batches or snapshots.

- The one-row-per-listing structure provides a suitable starting point for Silver-layer transformations and subsequent joins with Browse Search, subject to validating join cardinality and source consistency.


In [0]:
# =====================================================
# Column-level completeness profile
# =====================================================

total_rows = item_details_df.count()

null_profile = item_details_df.select([
    F.sum(
        F.when(F.col(c).isNull(), 1).otherwise(0)
    ).alias(c)
    for c in item_details_df.columns
])

null_counts = null_profile.first().asDict()

completeness_data = [
    (
        column,
        null_counts[column],
        round(
            (total_rows - null_counts[column]) / total_rows * 100,
            2
        )
    )
    for column in item_details_df.columns
]

completeness_df = spark.createDataFrame(
    completeness_data,
    ["column_name", "null_count", "completeness_pct"]
)

display(
    completeness_df
    .orderBy("completeness_pct", "column_name")
)


## 8. Column-Level Completeness Analysis

Column-level null profiling was performed across all 111 columns in the Item Details dataset to understand attribute availability and identify fields requiring special handling during Silver-layer transformations.

The completeness percentage represents the proportion of records in which a column contains a non-null value.

### 8.1 Highly Sparse Attributes

Several columns contain null values in nearly all records.

| Column | Null Count | Completeness (%) |
|---|---:|---:|
| `product_fiche_web_url` | 48,427 | 0.00 |
| `size_system` | 48,427 | 0.00 |
| `reserve_price_met` | 48,421 | 0.01 |
| `energy_efficiency_class` | 48,419 | 0.02 |
| `repair_score` | 48,412 | 0.03 |
| `marketing_price__original_price__value` | 48,411 | 0.04 |
| `manufacturer__contact_url` | 48,387 | 0.08 |
| `current_bid_price__value` | 48,386 | 0.09 |
| `minimum_price_to_bid__value` | 48,386 | 0.09 |
| `bid_count` | 48,355 | 0.15 |

These attributes have very limited coverage in the current snapshot. In particular, several bidding-related and manufacturer-related fields are populated for only a small subset of listings.

The completeness percentages are rounded in the profiling output. A displayed value of 0.00% does not necessarily mean that the column contains no non-null values.

### 8.2 Pricing and Product Attributes

Pricing-related fields exhibit varying levels of completeness.

| Column | Null Count | Completeness (%) |
|---|---:|---:|
| `marketing_price__discount_percentage` | 39,764 | 17.89 |
| `marketing_price__discount_amount__value` | 39,517 | 18.40 |
| `marketing_price__original_price__value` | 40,495 | 16.38 |
| `price__converted_from_value` | 43,934 | 9.28 |
| `unit_price__value` | 46,337 | 4.32 |
| `brand` | 11,055 | 77.17 |
| `mpn` | 38,538 | 20.42 |
| `gtin` | 16,927 | 65.05 |
| `color` | 2,601 | 46.27 |
| `material` | 3,140 | 35.14 |

The standard listing price fields (`price__value` and `price__currency`) are fully populated, whereas converted prices, discount information, and unit pricing have substantially lower coverage.

Product attributes such as brand and GTIN are available for a considerable proportion of listings, but their completeness varies by attribute.

This suggests that product-level analytical features should be selected based on actual field availability rather than assuming every listing contains all product attributes.

### 8.3 Product Reviews and Listing Information

| Column | Null Count | Completeness (%) |
|---|---:|---:|
| `primary_product_review_rating__average_rating` | 44,522 | 8.07 |
| `primary_product_review_rating__review_count` | 44,522 | 8.07 |
| `item_end_date` | 44,628 | 7.85 |
| `subtitle` | 45,025 | 7.03 |
| `short_description` | 4,804 | 90.08 |
| `description` | 101 | 99.79 |
| `image__image_url` | 27 | 99.94 |
| `seller__feedback_percentage` | 4 | 99.99 |

The main description and image URL fields have very high completeness. This provides broad availability of descriptive and visual listing information.

Product review ratings and review counts are available for a much smaller subset of listings. Their absence should not be interpreted as a zero rating or zero reviews without validating the source semantics.

The `item_end_date` field is also sparsely populated, so it should not be assumed to be available for calculating listing duration across the entire dataset.

### 8.4 Core Fields with Complete Coverage

The following important fields have no null values in the current snapshot:

| Column | Completeness (%) |
|---|---:|
| `_dlt_id` | 100 |
| `_dlt_load_id` | 100 |
| `item_id` | 100 |
| `legacy_item_id` | 100 |
| `title` | 100 |
| `condition` | 100 |
| `condition_id` | 100 |
| `category_id` | 100 |
| `category_path` | 100 |
| `item_creation_date` | 100 |
| `item_web_url` | 100 |
| `listing_marketplace_id` | 100 |
| `price__value` | 100 |
| `price__currency` | 100 |
| `seller__username` | 100 |
| `seller__feedback_score` | 100 |

These fields provide a consistent foundation for listing identification, product categorization, pricing analysis, seller analysis, and listing-level joins.

The completeness results establish that the core listing attributes are available for every record in this snapshot, while several optional or specialized attributes are populated only for subsets of listings.

### 8.5 Implications for Silver-Layer Processing

The completeness profile suggests the following handling strategy:

- **Retain core listing fields:** Preserve fully populated identifiers, titles, categories, prices, seller attributes, and listing creation dates.

- **Preserve sparse attributes where useful:** Fields such as review ratings, discount information, bidding metrics, and manufacturer details may still be valuable for specific analytical use cases.

- **Avoid blanket null removal:** Dropping records based on optional attribute nulls could remove valid listings and unnecessarily reduce the dataset.

- **Validate feature eligibility:** Before using a field in opportunity scoring or downstream analytics, examine its coverage and determine whether missing values have a meaningful business interpretation.

- **Separate missing values from business values:** A null review rating, for example, should not automatically be converted to zero. Missingness and an actual zero value represent different conditions.

The current profiling establishes field-level completeness but does not yet establish whether non-null values are valid, correctly formatted, or suitable for analytical calculations.


In [0]:
# =====================================================
# Check empty and whitespace-only strings
# =====================================================

string_columns = [
    field.name
    for field in item_details_df.schema.fields
    if field.dataType.simpleString() == "string"
]

blank_profile = item_details_df.agg(*[
    F.sum(
        F.when(
            F.col(c).isNotNull() &
            (F.trim(F.col(c)) == ""),
            1
        ).otherwise(0)
    ).alias(c)
    for c in string_columns
])

blank_counts = blank_profile.first().asDict()

blank_data = [
    (column, blank_counts[column])
    for column in string_columns
    if blank_counts[column] > 0
]

from pyspark.sql.types import (
    StructType,
    StructField,
    StringType,
    LongType
)

blank_schema = StructType([
    StructField("column_name", StringType(), False),
    StructField("blank_count", LongType(), False)
])

blank_df = spark.createDataFrame(
    blank_data,
    schema=blank_schema
)

display(blank_df.orderBy(F.desc("blank_count")))


## 9. Empty and Whitespace-Only String Checks

After profiling null values, a separate check was performed on string columns to identify empty strings and whitespace-only values.

This check complements the null profile because an empty string (`""`) or a string containing only spaces is not necessarily treated as a null value by Spark.

### 9.1 Findings

The check examined all columns with Spark's `StringType` data type.

| Check | Result |
|---|---|
| String columns examined | All string-type columns |
| Empty strings (`""`) | None detected |
| Whitespace-only strings | None detected |

The profiling returned no rows, indicating that none of the examined string columns contained non-null values that became empty after trimming whitespace.

### 9.2 Interpretation

- **No blank-string anomalies detected:** The check did not identify any empty or whitespace-only strings in the current Item Details snapshot.

- **Nulls remain a separate concern:** This does not mean that all string columns are fully populated. The previous completeness analysis identified several fields with substantial null values.

- **String cleaning requirements:** No additional cleaning is indicated specifically for empty or whitespace-only strings based on this check. Other validation, such as trimming leading or trailing spaces in non-empty values, has not yet been assessed.

### 9.3 Implications for Silver Processing

The current results suggest that empty-string normalization is not required as a corrective step for the examined snapshot.

However, null handling and field-specific validation will still be necessary for downstream transformations. Future ingestion batches should also be subject to the same checks, as source data characteristics may change over time.


In [0]:
# =====================================================
# Categorical distributions
# =====================================================

categorical_columns = [
    "condition",
    "condition_id",
    "listing_marketplace_id",
    "price__currency"
]

for column in categorical_columns:
    print(f"\n{'=' * 60}")
    print(f"Distribution: {column}")
    print("=" * 60)

    display(
        item_details_df
        .groupBy(column)
        .agg(
            F.count("*").alias("record_count")
        )
        .withColumn(
            "percentage",
            F.round(
                F.col("record_count") /
                item_details_df.count() * 100,
                2
            )
        )
        .orderBy(F.desc("record_count"))
    )


## 10. Categorical Distributions

The categorical distributions of `condition`, `condition_id`, `listing_marketplace_id`, and `price__currency` were examined to understand the composition of the Item Details dataset.

The results provide insight into product condition, marketplace coverage, and currency consistency.

### 10.1 Listing Condition

The `condition` column contains 20 distinct values in the current snapshot.

| Condition | Record Count | Percentage (%) |
|---|---:|---:|
| New | 35,212 | 72.71 |
| Brand New | 5,633 | 11.63 |
| New with tags | 2,363 | 4.88 |
| Open box | 1,431 | 2.95 |
| New without tags | 898 | 1.85 |
| New with box | 765 | 1.58 |
| New without box | 494 | 1.02 |
| New other (see details) | 352 | 0.73 |
| New with box and papers | 331 | 0.68 |
| New with packaging | 317 | 0.65 |
| New/Factory Sealed | 217 | 0.45 |
| New – Open box | 133 | 0.27 |
| New without packaging | 106 | 0.22 |
| New without box or papers | 53 | 0.11 |
| New with defects | 53 | 0.11 |
| New with imperfections | 45 | 0.09 |
| New: Other (See Details) | 19 | 0.04 |
| Like New | 4 | 0.01 |
| Used | 1 | 0.00 |
| Very Good | 1 | 0.00 |

**Findings:**

- The `New` category accounts for 72.71% of the dataset, making it the dominant condition label.
- `Brand New` represents another 11.63% of listings. Together, these two exact labels account for approximately 84.34% of the snapshot.
- Several condition labels describe similar states but use different wording, such as `New with tags`, `New without tags`, and `New/Factory Sealed`.
- The distribution is heavily concentrated in new-condition listings, while explicitly used and like-new listings are rare in this snapshot.

The condition labels should not be merged solely on the basis of similar wording. Any normalization should preserve distinctions that may be meaningful for product quality or buyer expectations.

### 10.2 Condition ID

The `condition_id` column contains six distinct values.

| Condition ID | Record Count | Percentage (%) |
|---|---:|---:|
| 1000 | 44,838 | 92.59 |
| 1500 | 3,486 | 7.20 |
| 1750 | 98 | 0.20 |
| 2750 | 4 | 0.01 |
| 3000 | 1 | 0.00 |
| 4000 | 1 | 0.00 |

**Findings:**

- Condition ID `1000` accounts for 92.59% of records.
- Condition ID `1500` accounts for another 7.20%.
- The remaining condition IDs collectively represent a very small proportion of the snapshot.
- The distribution is consistent with the concentration of new-condition listings observed in the `condition` column.

The numeric condition IDs should be interpreted using the source's condition definitions rather than treated as numerical measurements. Their values do not imply an ordered scale of product quality.

### 10.3 Listing Marketplace

The `listing_marketplace_id` distribution shows that the dataset contains listings from eight marketplace identifiers.

| Marketplace ID | Record Count | Percentage (%) |
|---|---:|---:|
| EBAY_US | 38,152 | 78.78 |
| EBAY_MOTORS_US | 5,782 | 11.94 |
| EBAY_DE | 1,732 | 3.58 |
| EBAY_AU | 1,613 | 3.33 |
| EBAY_GB | 818 | 1.69 |
| EBAY_CA | 200 | 0.41 |
| EBAY_IT | 128 | 0.26 |
| EBAY_FR | 3 | 0.01 |

**Findings:**

- The US marketplace (`EBAY_US`) accounts for 78.78% of all records.
- The US Motors marketplace contributes another 11.94%.
- The remaining records are distributed across Germany, Australia, the United Kingdom, Canada, Italy, and France.
- The dataset is therefore predominantly US-focused, with a smaller international component.

Marketplace identifiers should be retained for filtering and segmentation. Cross-marketplace price comparisons will also require consideration of currency and market-specific differences.

### 10.4 Price Currency

The `price__currency` column contains only one distinct value.

| Currency | Record Count | Percentage (%) |
|---|---:|---:|
| USD | 48,428 | 100.00 |

**Findings:**

- All 48,428 records have USD as their listing-price currency.
- No variation in `price__currency` was observed in the current snapshot.
- This provides a consistent currency basis for aggregating and comparing the primary listing prices within this dataset.

However, the presence of international marketplace identifiers alongside USD prices means that currency consistency should not be interpreted as evidence that all listings originate from the US.

The original and converted currency fields should still be examined separately before performing any currency conversion or cross-marketplace analysis.

### 10.5 Implications for Downstream Processing

The categorical distributions establish several considerations for the Silver and Gold layers:

- Preserve the original condition labels and condition IDs before applying any standardized condition mapping.
- Retain marketplace identifiers to support marketplace-level segmentation and filtering.
- Use USD as the observed currency of the current primary-price field, while validating other pricing fields independently.
- Avoid interpreting categorical identifiers as continuous numerical variables.
- Reassess these distributions when new ingestion batches are added, since the current results describe only this snapshot.


In [0]:
# =====================================================
# Category distribution
# =====================================================

display(
    item_details_df
    .groupBy("category_id", "category_path")
    .agg(
        F.count("*").alias("record_count")
    )
    .withColumn(
        "percentage",
        F.round(
            F.col("record_count") / item_details_df.count() * 100,
            2
        )
    )
    .orderBy(F.desc("record_count"))
    .limit(50)
)


## 11. Category Distribution

The Item Details dataset was grouped by `category_id` and `category_path` to examine the distribution of listings across product categories.

The aggregation calculates the number of records and the percentage of the total dataset represented by each category.

The output was ordered by record count in descending order, with the top 50 categories exported to CSV for further inspection.

### 11.1 Category Distribution — Top 50

The complete top-50 category distribution is available in the exported file:

`category.csv`

The file contains the following fields:

| Column | Description |
|---|---|
| `category_id` | eBay category identifier |
| `category_path` | Hierarchical category path |
| `record_count` | Number of listings in the category |
| `percentage` | Percentage of total Item Details records |

Each category percentage is calculated as:

\[
\text{Category Percentage} =
\frac{\text{Category Record Count}}{\text{Total Records}} \times 100
\]

The denominator is 48,428 records, representing the complete Item Details snapshot.

### 11.2 Categories with the Highest Record Counts

The following categories appear at the top of the exported distribution:

| Category | Record Count | Percentage (%) |
|---|---:|---:|
| Video Games & Consoles \| Video Games | 1,629 | 3.36 |
| Movies & TV \| DVDs & Blu-ray Discs | 1,341 | 2.77 |
| Books & Magazines \| Books | 1,116 | 2.30 |
| Toys & Hobbies \| Action Figures & Accessories \| Action Figures | 975 | 2.01 |
| Business & Industrial \| Industrial Automation & Motion Controls \| PLCs & HMIs \| PLC Processors | 856 | 1.77 |
| Music \| CDs | 755 | 1.56 |
| Toys & Hobbies \| Diecast & Toy Vehicles \| Cars, Trucks & Vans \| Contemporary Manufacture | 742 | 1.53 |
| Clothing, Shoes & Accessories \| Women \| Women's Bags & Handbags | 696 | 1.44 |
| Business & Industrie \| Automation, Antriebe & Motoren \| Sonstiges Automations Equipment | 678 | 1.40 |
| Music \| Vinyl Records | 626 | 1.29 |

The largest individual category accounts for 3.36% of the dataset. No single category dominates the snapshot, indicating that the listings are distributed across numerous product categories.

### 11.3 Category Diversity and Marketplace Coverage

The category distribution includes several broad product groups:

- **Electronics and computing:** Computer accessories, laptop cases, tablet accessories, keyboards, and motherboards appear among the categories.

- **Entertainment and collectibles:** Video games, DVDs, books, CDs, vinyl records, action figures, and diecast vehicles are represented.

- **Business and industrial products:** Industrial automation equipment, PLC processors, and related machinery components appear in the results.

- **Other consumer products:** Clothing, watches, kitchen storage, camera accessories, and automotive parts are also present.

The distribution therefore extends beyond electronics and computing products.

This is an important observation for the Market Intelligence Platform because the current Item Details snapshot contains a broader category mix than an electronics-only analytical scope would imply.

### 11.4 Category Hierarchy and Naming

The `category_path` field provides a hierarchical representation of the source category.

For example:

`Computers/Tablets & Networking | Computer Components & Parts | Motherboards`

This allows listings to be analyzed at different levels of the category hierarchy, rather than relying exclusively on individual category IDs.

The output also contains category paths in different languages, such as English and German. For example, a Business & Industrial category appears with the German path `Business & Industrie`.

This should be considered when standardizing category labels or combining category information across marketplaces.

Category IDs and their corresponding paths should be preserved during Silver-layer processing. Any translation, normalization, or mapping to a common category taxonomy should be performed through an explicit mapping rather than by modifying the source values directly.

### 11.5 Implications for Downstream Processing

- **Category-based filtering:** The category hierarchy can be used to select listings relevant to the intended product scope.

- **Category-level aggregation:** Record counts and pricing metrics can be aggregated at category ID or higher category levels.

- **Category normalization:** Differences in language and naming should be investigated before combining categories across marketplaces.

- **Scope validation:** The broad category distribution should be compared against the intended Market Intelligence Platform scope before building Gold-layer analytical tables.

The exported CSV provides the detailed top-50 distribution for subsequent category mapping, filtering, and analytical exploration.


In [0]:
# Top-level category distribution

category_distribution = (
    item_details_df
    .withColumn(
        "top_level_category",
        F.split(F.col("category_path"), r"\|").getItem(0)
    )
    .groupBy("top_level_category")
    .agg(
        F.count("*").alias("record_count"),
        F.countDistinct("category_id").alias("distinct_category_ids")
    )
    .withColumn(
        "percentage",
        F.round(
            F.col("record_count") / item_details_df.count() * 100,
            2
        )
    )
    .orderBy(F.desc("record_count"))
)

display(category_distribution)


### 12.1 Top-Level Category Distribution — Findings

The top-level category aggregation contains 74 distinct category labels, accounting for all 48,428 records in the Item Details dataset.

The distribution is ordered by record count in descending order. The following categories represent the largest portions of the collected listings.

| Top-Level Category | Record Count | Percentage (%) | Distinct Category IDs |
|---|---:|---:|---:|
| Computers/Tablets & Networking | 6,186 | 12.77 | 130 |
| eBay Motors | 5,782 | 11.94 | 539 |
| Home & Garden | 4,420 | 9.13 | 559 |
| Toys & Hobbies | 3,797 | 7.84 | 212 |
| Video Games & Consoles | 3,134 | 6.47 | 28 |
| Collectibles | 3,113 | 6.43 | 498 |
| Clothing, Shoes & Accessories | 2,647 | 5.47 | 131 |
| Business & Industrial | 2,417 | 4.99 | 412 |
| Jewelry & Watches | 1,586 | 3.27 | 58 |
| Books & Magazines | 1,458 | 3.01 | 11 |

### 12.2 Distribution Across Product Domains

Computers/Tablets & Networking is the largest top-level category, accounting for 12.77% of the dataset, followed by eBay Motors at 11.94%.

Home & Garden contributes another 9.13%, while Toys & Hobbies accounts for 7.84%.

The top five categories together represent approximately 48.15% of all records. This indicates that nearly half of the snapshot is concentrated in five broad product domains, while the remaining records are distributed across other categories.

The dataset is therefore not limited to computer hardware, despite Computers/Tablets & Networking being the largest individual category.

### 12.3 Category Breadth and Hierarchy

The number of distinct category IDs varies substantially across top-level categories.

For example, Home & Garden contains 559 distinct category IDs, while Video Games & Consoles contains only 28. eBay Motors also covers a broad range, with 539 distinct category IDs.

This shows that the number of listings and the breadth of the category hierarchy are separate characteristics.

A category with a large record count does not necessarily contain the largest number of distinct subcategories. Both measures should be retained for downstream category-level analysis.

### 12.4 Category Naming and Standardization

The output contains category labels in multiple languages, including English, German, French, and Italian.

Examples include `Jouets et jeux`, `Fotografia e video`, and `Ordinateurs/tablettes, réseaux`.

These labels may represent equivalent or related product domains across different marketplace taxonomies. However, they should not be combined solely based on translated names without validating their underlying category IDs and hierarchy.

For the Silver layer, preserve the original `category_id` and `category_path` values. A standardized top-level category can subsequently be introduced through an explicit category mapping.

### 12.5 Implications for the Market Intelligence Platform

The top-level distribution confirms that the collected snapshot contains a broad marketplace product mix.

For the current electronics-focused MVP, Computers/Tablets & Networking provides a natural starting point for category filtering. However, the broader dataset includes automotive, home, entertainment, collectibles, and other product domains.

Category filtering should therefore be applied explicitly before generating electronics-focused Silver and Gold tables.

The original category information should remain available so that future analytical scope can be expanded without losing the source taxonomy.


In [0]:
# =====================================================
# Numeric business field profiling
# =====================================================

numeric_columns = [
    "price__value",
    "seller__feedback_score",
    "seller__feedback_percentage",
    "primary_product_review_rating__average_rating",
    "primary_product_review_rating__review_count",
]

for column in numeric_columns:

    print(f"\n{'=' * 60}")
    print(f"Statistics: {column}")
    print("=" * 60)

    display(
        item_details_df
        .select(
            F.col(column).cast("double").alias(column)
        )
        .summary(
            "count",
            "min",
            "25%",
            "50%",
            "75%",
            "max",
            "mean",
            "stddev"
        )
    )


## 13. Numeric Business Field Profiling

The main numeric business fields were profiled using descriptive statistics to understand their range, central tendency, spread, and potential outliers.

The fields examined were:

- `price__value`
- `seller__feedback_score`
- `seller__feedback_percentage`
- `primary_product_review_rating__average_rating`
- `primary_product_review_rating__review_count`

The statistics include count, minimum, quartiles, maximum, mean, and standard deviation.

### 13.1 Listing Price

| Statistic | `price__value` |
|---|---:|
| Count | 48,428 |
| Minimum | 0.55 |
| 25th Percentile | 16.98 |
| Median | 31.49 |
| 75th Percentile | 71.99 |
| Maximum | 250,000.00 |
| Mean | 155.1964 |
| Standard Deviation | 1,390.3960 |

The listing price ranges from **$0.55 to $250,000**, with a median price of **$31.49**.

The mean price of approximately **$155.20** is substantially higher than the median. This difference indicates a strongly right-skewed price distribution, where a relatively small number of high-priced listings pull the mean upward.

The maximum value of $250,000 is substantially higher than the 75th percentile of $71.99. This indicates the presence of extreme high-value listings that should be investigated before using the raw mean for analytical reporting.

For downstream opportunity scoring, price should therefore be treated carefully. Median-based statistics, percentile-based thresholds, or explicitly defined outlier handling may be more representative than relying only on the arithmetic mean.

---

### 13.2 Seller Feedback Score

| Statistic | `seller__feedback_score` |
|---|---:|
| Count | 48,428 |
| Minimum | -1.0 |
| 25th Percentile | 426.0 |
| Median | 1,676.0 |
| 75th Percentile | 8,829.0 |
| Maximum | 4,541,062.0 |
| Mean | 70,440.03 |
| Standard Deviation | 236,827.41 |

Seller feedback scores show substantial variation across listings.

The median seller feedback score is **1,676**, while the mean is approximately **70,440**. The large difference between these values indicates a heavily right-skewed distribution.

The maximum observed value is **4,541,062**, which is considerably larger than the 75th percentile of 8,829. A relatively small number of sellers therefore have extremely large feedback counts.

The minimum value of **-1** is also notable and should be investigated against the source-field semantics before using this field in scoring or filtering logic.

For seller-quality analysis, the raw feedback score should not be interpreted in isolation. Its extreme skew means that percentile-based transformations or logarithmic scaling may be useful when the field is eventually used as an analytical feature.

---

### 13.3 Seller Feedback Percentage

| Statistic | `seller__feedback_percentage` |
|---|---:|
| Count | 48,424 |
| Minimum | 0.0 |
| 25th Percentile | 99.5 |
| Median | 99.8 |
| 75th Percentile | 100.0 |
| Maximum | 100.0 |
| Mean | 97.9380 |
| Standard Deviation | 12.6482 |

Seller feedback percentage is available for 48,424 of the 48,428 records, meaning four records are missing this value.

The distribution is concentrated near the upper end of the scale. The median is **99.8%**, while the 75th percentile and maximum are both **100%**.

The mean of approximately **97.94%** is lower than the median because a smaller number of listings have substantially lower feedback percentages.

This field therefore has a very different distribution from `seller__feedback_score`: feedback percentage is tightly concentrated near 100%, while feedback score has a highly skewed long tail.

For seller-quality analysis, the two fields should be treated as separate signals rather than interchangeable measures.

---

### 13.4 Product Review Average Rating

| Statistic | `primary_product_review_rating__average_rating` |
|---|---:|
| Count | 3,906 |
| Minimum | 0.0 |
| 25th Percentile | 0.0 |
| Median | 3.67 |
| 75th Percentile | 4.90 |
| Maximum | 5.0 |
| Mean | 2.4586 |
| Standard Deviation | 2.3949 |

Product review ratings are available for only **3,906 records**, compared with 48,428 total listings.

This means the review-rating field is populated for only a subset of the dataset. The previous completeness analysis also showed that this field has approximately **8.07% coverage**.

The median rating is **3.67**, while the 75th percentile is **4.90** and the maximum is **5.0**.

The presence of a minimum and first quartile of **0.0** should be investigated before treating zero as a genuine product rating. It may represent a source-specific value or another condition rather than a conventional customer rating.

Because review ratings are available for only a small subset of listings, this field should not be used as a mandatory feature for all listings without an explicit missing-value strategy.

---

### 13.5 Product Review Count

| Statistic | `primary_product_review_rating__review_count` |
|---|---:|
| Count | 3,906 |
| Minimum | 0.0 |
| 25th Percentile | 0.0 |
| Median | 1.0 |
| 75th Percentile | 7.0 |
| Maximum | 1,671.0 |
| Mean | 16.3451 |
| Standard Deviation | 73.1349 |

The product review count has the same population size as the average-rating field: **3,906 listings**.

The median listing has only **1 review**, while the 75th percentile is **7 reviews**. The mean of approximately **16.35 reviews** is substantially higher than the median, indicating a right-skewed distribution.

The maximum review count is **1,671**, showing that a small number of products have substantially more reviews than the typical listing.

The combination of low median review count and high maximum suggests that review volume is highly uneven across products.

This field may be useful as a product-popularity signal, but its skew should be considered when constructing opportunity-scoring features.

---

### 13.6 Patterns Across the Numeric Fields

Several common patterns emerge from the profiling:

| Field | Main Distribution Characteristic |
|---|---|
| `price__value` | Strong right skew with extreme high-value listings |
| `seller__feedback_score` | Extremely right-skewed with very large seller counts |
| `seller__feedback_percentage` | Highly concentrated near 100% |
| `primary_product_review_rating__average_rating` | Sparse and bounded between 0 and 5 |
| `primary_product_review_rating__review_count` | Sparse and strongly right-skewed |

The price and seller feedback score distributions demonstrate why mean values alone may be misleading for business analytics.

The review-related fields have an additional challenge: they are available for only a small subset of listings.

These characteristics should be considered before converting the fields into standardized analytical features.

### 13.7 Data Engineering Considerations

The profiling suggests several checks for the Silver layer:

- `price__value` should be converted from string to an appropriate numeric type after validating its source format.

- `seller__feedback_score` should be investigated for the `-1` value and extreme upper-tail values before feature engineering.

- `seller__feedback_percentage` should be validated against its expected percentage range.

- Product review fields should retain their null state unless the source semantics justify another representation.

- Highly skewed fields such as price, feedback score, and review count may require percentile-based or logarithmic transformations for downstream scoring models.

At this stage, these observations describe the distributions only. No outlier values have been removed or transformed yet.

In [0]:
# =====================================================
# Investigate unusual numeric values
# =====================================================

checks = {
    "seller_feedback_score_-1": (
        F.col("seller__feedback_score") == -1
    ),

    "seller_feedback_percentage_0": (
        F.col("seller__feedback_percentage").cast("double") == 0
    ),

    "review_rating_0": (
        F.col("primary_product_review_rating__average_rating").cast("double") == 0
    ),

    "review_count_0": (
        F.col("primary_product_review_rating__review_count") == 0
    )
}

for name, condition in checks.items():

    print(f"\n{name}")

    display(
        item_details_df
        .filter(condition)
        .select(
            "item_id",
            "title",
            "seller__username",
            "seller__feedback_score",
            "seller__feedback_percentage",
            "primary_product_review_rating__average_rating",
            "primary_product_review_rating__review_count"
        )
        .limit(20)
    )

## 14. Unusual Numeric Values and Source Semantics

The numeric profiling identified several unusual values that required record-level investigation before deciding whether they should be treated as invalid data.

### `seller__feedback_score = -1`

Two listings were found with `seller__feedback_score = -1`.

Both records had:

- `seller__feedback_percentage = 0.0`
- No product review information
- `seller__feedback_score = -1`

A negative feedback count does not have an obvious business interpretation. The repeated combination of `-1`, `0.0` feedback percentage, and missing review information suggests that `-1` may be a special source/API value representing an unavailable or unknown seller-feedback state.

The exact source semantics have not yet been confirmed.

**Current handling:**

- Preserve `-1` in the Bronze layer.
- Do not convert `-1` to `0`.
- Investigate the source/API semantics before transforming it.
- If confirmed to represent an unavailable value, it can be converted to `NULL` in Silver.

---

### `seller__feedback_percentage = 0.0`

Inspection of individual records showed that `0.0` does not necessarily mean that a seller has zero feedback.

Examples found in the data:

| Seller Feedback Score | Feedback Percentage |
|---:|---:|
| 0 | 0.0 |
| 2 | 0.0 |
| 9 | 0.0 |
| 59 | 0.0 |

A seller with a feedback score of `59` can therefore have a feedback percentage of `0.0`.

This means the field should not currently be treated as a straightforward seller-quality score without understanding how eBay represents, calculates, or rounds this value.

**Current handling:**

- Preserve the raw `0.0` value.
- Do not automatically convert it to `NULL`.
- Do not interpret `0.0` as poor seller quality.
- Investigate the source semantics before using this field in scoring logic.

---

### `primary_product_review_rating__average_rating = 0`

This field shows a clearer pattern.

The exported records repeatedly contain:

| Field | Value |
|---|---:|
| `average_rating` | 0 |
| `review_count` | 0 |

The combination of `average_rating = 0` and `review_count = 0` appears to represent the absence of product review data rather than a genuine zero-star rating.

Therefore:

```text
average_rating = 0
review_count = 0
        ↓
No product review data available
```
This distinction is important for future opportunity scoring. A product with no reviews should not automatically be interpreted as a product with a poor rating.

Potential Silver handling:
- Preserve `review_count = 0`.
- Preserve the original raw values in Bronze.
- Consider converting `average_rating = 0` to NULL in Silver when `review_count = 0`.
- Use the transformed field only after the source semantics have been validated.

In [0]:
# =====================================================
# Date/time profiling
# =====================================================

date_columns = [
    "item_creation_date",
    "item_end_date"
]

for column in date_columns:

    print(f"\n{'=' * 60}")
    print(f"Date profile: {column}")
    print("=" * 60)

    date_profile = (
        item_details_df
        .select(
            F.col(column),
            F.to_timestamp(F.col(column)).alias("parsed_timestamp")
        )
        .agg(
            F.count("*").alias("total_rows"),
            F.count(column).alias("non_null"),
            F.min("parsed_timestamp").alias("min_timestamp"),
            F.max("parsed_timestamp").alias("max_timestamp"),
            F.sum(
                F.when(
                    F.col(column).isNotNull() &
                    F.col("parsed_timestamp").isNull(),
                    1
                ).otherwise(0)
            ).alias("parse_failures")
        )
    )

    display(date_profile)

In [0]:
# =====================================================
# Validate listing lifecycle timestamps
# =====================================================

date_validation = (
    item_details_df
    .select(
        "item_id",
        "item_creation_date",
        "item_end_date",
        F.to_timestamp("item_creation_date").alias("creation_ts"),
        F.to_timestamp("item_end_date").alias("end_ts")
    )
    .filter(
        F.col("end_ts").isNotNull()
    )
    .withColumn(
        "invalid_date_order",
        F.col("end_ts") < F.col("creation_ts")
    )
)

display(
    date_validation
    .groupBy("invalid_date_order")
    .agg(
        F.count("*").alias("record_count")
    )
)

## 15. Date/Time Profiling

The listing date fields were profiled to check timestamp completeness, parsing reliability, valid date ranges, and the chronological relationship between listing creation and end dates.

The fields examined were:

- `item_creation_date`
- `item_end_date`

### 15.1 `item_creation_date`

| Metric | Result |
|---|---:|
| Total Rows | 53,310 |
| Non-Null Values | 53,310 |
| Null Values | 0 |
| Parse Failures | 0 |
| Minimum Timestamp | `2026-09-15T20:10:14+00:00` |
| Maximum Timestamp | `2026-09-19T00:00:00+00:00` |

All 53,310 records contain a non-null `item_creation_date`.

The timestamp conversion produced **zero parse failures**, indicating that all observed values could be successfully parsed into timestamps.

The creation timestamps range from **September 15, 2026** through **September 19, 2026** in the current snapshot.

This makes `item_creation_date` a consistently available field for listing-age calculations and other time-based analysis.

---

### 15.2 `item_end_date`

| Metric | Result |
|---|---:|
| Total Rows | 53,310 |
| Non-Null Values | 4,508 |
| Null Values | 48,802 |
| Parse Failures | 0 |
| Minimum Timestamp | `2026-09-18T16:52:22+00:00` |
| Maximum Timestamp | `2026-09-30T03:57:49+00:00` |

`item_end_date` is substantially less complete than `item_creation_date`.

Only **4,508 of 53,310 records** contain a non-null end timestamp, while **48,802 records** have a null value.

Among the non-null values, timestamps range from **September 18, 2026** through **September 30, 2026**.

No timestamp parsing failures were detected. Therefore, the primary issue with this field is its availability rather than its timestamp format.

The null values should not automatically be treated as invalid records because the absence of an end date may depend on the listing type or current listing state.

---

### 15.3 Date Relationship Validation

A chronological validation was performed on the 4,508 records where `item_end_date` is available.

All 4,508 listings satisfy:

```text
item_end_date >= item_creation_date
```
Therefore:
| Validation | Result |
|---|---:|
| Listings with `item_end_date` | 4,508 |
| Invalid date sequences | 0 |
| Timestamp parsing failures | 0 |
| `item_creation_date` populated | Yes |
| `item_end_date` optional | Yes |
| Non-null `item_end_date` values chronologically valid | Yes |

There are no observed cases where a listing ends before its creation timestamp.
This means no date-order correction is required for the current snapshot.

15.4 Date Field Comparison
| Field | Non-Null | Null | Parse Failures | Chronological Validation |
|---|---:|---:|---:|---|
| `item_creation_date` | 53,310 | 0 | 0 | Fully populated |
| `item_end_date` | 4,508 | 48,802 | 0 | Valid when present |

The two fields therefore have different completeness characteristics.
`item_creation_date` is fully populated and consistently parseable, while item_end_date is optional but valid when present.

The current profiling provides evidence that the date fields can be safely converted to Spark timestamp types without requiring special parsing or date-order correction logic.

15.5 Silver-Layer Handling
- The planned Silver-layer representation is:
```text
item_creation_date → timestamp
item_end_date      → nullable timestamp
```

`item_creation_date` should be converted directly to a proper timestamp type.

`item_end_date` should also be converted to a timestamp, while preserving its null values for listings where the source does not provide an end date.

No artificial values should be introduced for missing end dates.

No special date-order correction is currently required because every observed non-null `item_end_date` is greater than or equal to its corresponding `item_creation_date`.

The raw date values should remain available in Bronze for traceability, while the Silver layer can expose the standardized timestamp representation.